# EU CRR RAG — Ingestion Pipeline (GPU)

Run this notebook on a **T4 GPU** runtime (Runtime → Change runtime type → T4 GPU).

**Before running:**
1. Upload `crr_raw_en.html` and `crr_raw_ita.html` using the file browser (folder icon on the left)
2. Fill in your API keys in the _Secrets_ cell below

**Expected runtime:** ~30–60 min for EN + IT on a T4 GPU.

## 0 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Switch runtime to T4 GPU and re-run.')

## 1 — Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/eu-crr-rag.git"  # ← update this
REPO_DIR = "/content/eu-crr-rag"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

## 2 — Install dependencies

BGE-M3 is ~570 MB — this cell takes 2–5 min.

In [ ]:
# Install PyTorch with CUDA first to ensure FlagEmbedding picks up the GPU
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

# Install project deps (excludes fastapi/uvicorn since we only need the ingestion path)
!pip install -q \
    python-dotenv \
    llama-index-core \
    llama-index-embeddings-huggingface \
    llama-index-llms-openai \
    llama-index-vector-stores-qdrant \
    qdrant-client \
    "sentence-transformers>=3.0.0" \
    "FlagEmbedding>=1.2.0" \
    "transformers>=4.40.0,<5.0.0" \
    beautifulsoup4 \
    lxml \
    requests \
    openai

print('Install complete.')

## 3 — API keys

Enter your keys below. They are written to a `.env` file in the repo root (not committed).

In [ ]:
OPENAI_API_KEY    = "sk-..."                          # ← required
QDRANT_URL        = "https://your-cluster.qdrant.io"  # ← required
QDRANT_API_KEY    = "your-qdrant-api-key"             # ← required
LLAMA_CLOUD_API_KEY = ""                              # ← optional

env_lines = [
    f"OPENAI_API_KEY={OPENAI_API_KEY}",
    f"QDRANT_URL={QDRANT_URL}",
    f"QDRANT_API_KEY={QDRANT_API_KEY}",
]
if LLAMA_CLOUD_API_KEY:
    env_lines.append(f"LLAMA_CLOUD_API_KEY={LLAMA_CLOUD_API_KEY}")

with open(".env", "w") as f:
    f.write("\n".join(env_lines) + "\n")

print(".env written.")

## 4 — Upload HTML files

Upload `crr_raw_en.html` and `crr_raw_ita.html` using the Colab file browser **or** run the cell below to upload interactively.

In [ ]:
from google.colab import files
import shutil, os

print("Select your HTML files (crr_raw_en.html and crr_raw_ita.html):")
uploaded = files.upload()

for fname in uploaded:
    dest = os.path.join(REPO_DIR, fname)
    shutil.move(fname, dest)
    print(f"Moved {fname} → {dest}")

# Verify
for expected in ["crr_raw_en.html", "crr_raw_ita.html"]:
    path = os.path.join(REPO_DIR, expected)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  ✓ {expected} ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ {expected} NOT FOUND — upload it before continuing")

## 5 — Warm up BGE-M3

Downloads the model (~570 MB) on first run. Subsequent runs use the Colab disk cache.

In [ ]:
import torch
from FlagEmbedding import BGEM3FlagModel

print("Loading BGE-M3 (first run downloads ~570 MB)...")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
device = next(model.model.parameters()).device
print(f"Model loaded on: {device}")

# Quick smoke test
test_out = model.encode(["Article 92 own funds requirements"], return_dense=True, return_sparse=True)
print(f"Dense embedding dim: {len(test_out['dense_vecs'][0])}")
print("BGE-M3 OK.")

## 6 — Ingest English

`--reset` wipes the entire Qdrant collection first. Expected: 745 documents.

In [ ]:
!python -m src.pipelines.ingest_pipeline \
    --reset \
    --language en \
    --file crr_raw_en.html

## 7 — Ingest Italian

No `--reset` — appends to the existing collection. Expected: +745 documents.

In [ ]:
!python -m src.pipelines.ingest_pipeline \
    --language it \
    --file crr_raw_ita.html

## 8 — Validate

Expected item count: **~1490** (745 EN + 745 IT, no sub-chunking).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from src.indexing.vector_store import VectorStore

vs = VectorStore()
vs.connect()
count = vs.item_count
print(f"Items in Qdrant collection: {count}")

if count >= 1480:
    print("PASS — item count is in expected range (>= 1480)")
elif count >= 740:
    print("PARTIAL — only one language ingested; re-run Step 7 for Italian")
else:
    print(f"FAIL — unexpectedly low count ({count}); check logs above")

## 9 — Smoke-test a query

Verifies that the QueryEngine loads and returns a substantive answer citing article numbers.

In [ ]:
from src.query.query_engine import QueryEngine

engine = QueryEngine()
engine.load()

result = engine.query(
    "What are the own funds requirements under Article 92?",
    language="en"
)

print("=" * 60)
print(result.answer)
print("=" * 60)
print(f"\nSources returned: {len(result.sources)}")
for s in result.sources:
    article = s.get('article', '?')
    score   = s.get('score', 0)
    lang    = s.get('language', '?')
    print(f"  Article {article} [{lang}]  score={score:.3f}")

## Done

The Qdrant collection is now populated with ~1490 items (EN + IT, article-level, no sub-chunking).
Your local API server will use this index on next start — no re-ingest needed locally.